<div style="background: linear-gradient(135deg, #1a3a5c 0%, #2d6a9f 100%);
                    padding: 40px 50px; border-radius: 10px; margin-bottom: 10px;">
  <h1 style="color:#ffffff; font-family:'Georgia',serif; font-size:2.1em;
              font-weight:700; margin:0 0 12px 0; letter-spacing:0.5px; line-height:1.35;">
      Balancing Transparency and Predictive Power:<br>
      A Multi-Objective Framework for Explainable Crop Recommendation Systems
  </h1>

</div>
<div style="background:#f0f4f8; border-left:5px solid #2d6a9f;
            padding:14px 22px; border-radius:0 6px 6px 0; margin-top:6px;
            font-family:'Arial',sans-serif; color:#333; font-size:0.95em; line-height:1.7;">
  <b>Abstract:</b>
  This notebook presents a reproducible, end-to-end analytical pipeline for an explainable
  crop recommendation system. Four machine learning classifiers, Random Forest, XGBoost,
  Support Vector Machine, and Artificial Neural Network, are evaluated against both
  predictive performance and a structured explainability rubric. A novel
  <i>Multi-Objective Composite Score (MOCS)</i> is introduced to quantify the Pareto
  trade-off between accuracy and transparency, enabling principled model selection for
  farmer-facing advisory systems where trust and precision must co-exist.
</div>

---
## Table of Contents
| § | Section | Description |
|:-:|:--------|:------------|
| 1 | [Environment Setup & Configuration](#sec1) | Libraries, seeds, colour palettes, feature maps |
| 2 | [Exploratory Data Analysis](#sec2) | Distribution, correlation, PCA, feature engineering |
| 3 | [Pre-processing & Feature Engineering](#sec3) | Encoding, scaling, train/test split |
| 4 | [Multi-Objective Model Training](#sec4) | RF · XGBoost · SVM · ANN |
| 5 | [Comparative Performance Evaluation](#sec5) | Metrics table, bar chart, cross-validation |
| 6 | [Global Explainability — SHAP](#sec6) | TreeExplainer, permutation importance, consensus plot |
| 7 | [Local Explainability — LIME](#sec7) | Instance explanations, cross-model stability |
| 8 | [Multi-Objective Trade-off Analysis](#sec8) | XAI rubric, MOCS, Pareto bubble chart |
| 9 | [Statistical Significance Testing](#sec9) | Friedman test, Wilcoxon pairs, Cohen's *d* |
| 10 | [Conclusions & Summary Dashboard](#sec10) | Narrative summary, final 6-panel dashboard |

---
<a id="sec1"></a>
## Section 1 — Environment Setup & Configuration

This section initialises all third-party libraries, establishes a global random seed for
reproducibility, and defines shared aesthetic constants (colour palettes, feature label maps,
and the Plotly template) used consistently throughout the analysis.

In [24]:
! pip install lime

In [25]:
! pip install plotly

In [26]:
! pip install xgboost
! python -m pip install --upgrade pip

In [27]:
import shap
print(shap.__version__)

! pip install shap


0.51.0


In [28]:

import numpy as np
import pandas as pd
import warnings, time, json


In [29]:
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA
from itertools import combinations
from scipy import stats

from xgboost import XGBClassifier
#import shap
from lime import lime_tabular

SEED = 42
np.random.seed(SEED)

MODEL_COLORS = {
    'Random Forest': '#2ecc71',
    'XGBoost':       '#3498db',
    'SVM':           '#e67e22',
    'ANN':           '#9b59b6'
}

CAT_COLORS = {
    'Cereals':               '#F39C12',
    'Pulses':                '#27AE60',
    'Fruits':                '#E74C3C',
    'Melons & Cucurbits':    '#16A085',
    'Plantation Crops':      '#8E44AD',
    'Fiber & Industrial':    '#34495E'
}

PLOTLY_TEMPLATE = 'plotly_white'

groups = {
    'Cereals':            ['rice', 'maize'],
    'Pulses':             ['chickpea', 'kidneybeans', 'pigeonpeas',
                           'mothbeans', 'mungbean', 'blackgram', 'lentil'],
    'Fruits':             ['banana', 'mango', 'apple', 'orange',
                           'papaya', 'pomegranate', 'grapes'],
    'Melons & Cucurbits': ['watermelon', 'muskmelon'],
    'Plantation Crops':   ['coconut', 'coffee'],
    'Fiber & Industrial': ['cotton', 'jute']
}

crop_to_cat = {crop: cat for cat, crops in groups.items() for crop in crops}
cat_names   = list(groups.keys())

features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

feature_labels = {
    'N':           'Nitrogen (N)',
    'P':           'Phosphorus (P)',
    'K':           'Potassium (K)',
    'temperature': 'Temperature (°C)',
    'humidity':    'Humidity (%)',
    'ph':          'Soil pH',
    'rainfall':    'Rainfall (mm)'
}

print("Environment initialised successfully.")
print(f"  Plotly {__import__('plotly').__version__} | SHAP {shap.__version__}")

Environment initialised successfully.
  Plotly 6.6.0 | SHAP 0.51.0


---
<a id="sec2"></a>
## Section 2 — Exploratory Data Analysis

The exploratory phase characterises the dataset's structure, class balance, and
inter-feature relationships. Seven agrochemical and climatic variables serve as
predictors for 22 crop classes, which are further grouped into six agronomic
categories to facilitate higher-level pattern recognition.

### 2.1 Dataset Overview

In [30]:
# Load and annotate the dataset with agronomic category labels
df = pd.read_csv('Crop_recommendation.csv')
df['category'] = df['label'].map(crop_to_cat)

print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"  Rows           : {df.shape[0]:,}")
print(f"  Features       : {df.shape[1] - 2}  (excl. label & category)")
print(f"  Crop classes   : {df['label'].nunique()}")
print(f"  Categories     : {df['category'].nunique()}")
print(f"  Missing values : {df.isnull().sum().sum()}")
print(f"  Duplicates     : {df.duplicated().sum()}")
print()
display(df.describe().round(3))

DATASET OVERVIEW
  Rows           : 2,200
  Features       : 7  (excl. label & category)
  Crop classes   : 22
  Categories     : 6
  Missing values : 0
  Duplicates     : 0



,N,P,K,temperature,humidity,ph,rainfall
count,2200.000,2200.000,2200.000,2200.000,2200.000,2200.000,2200.000
mean,50.552,53.363,48.149,25.616,71.482,6.469,103.464
std,36.917,32.986,50.648,5.064,22.264,0.774,54.958
min,0.000,5.000,5.000,8.826,14.258,3.505,20.211
25%,21.000,28.000,20.000,22.769,60.262,5.972,64.552
50%,37.000,51.000,32.000,25.599,80.473,6.425,94.868
75%,84.250,68.000,49.000,28.562,89.949,6.924,124.268
max,140.000,145.000,205.000,43.675,99.982,9.935,298.560


### 2.2 Class and Category Distribution

In [31]:
# Pie chart — relative frequency of each agronomic category
fig = make_subplots(
    rows=1, cols=1,
    subplot_titles=('Crop Category Distribution',),
    specs=[[{'type': 'pie'}]]
)

cat_counts = df['category'].value_counts()

fig.add_trace(
    go.Pie(
        labels=cat_counts.index,
        values=cat_counts.values,
        marker=dict(
            colors=[CAT_COLORS[c] for c in cat_counts.index],
            line=dict(color='white', width=3)
        ),
        textinfo='label+percent',
        hovertemplate='<b>%{label}</b><br>Samples: %{value}<br>%{percent}<extra></extra>',
        pull=[0.05]*len(cat_counts)
    ),
    row=1, col=1
)

fig.update_layout(
    title=dict(
        text='<b>Figure 1 — Dataset Distribution by Crop Category</b>',
        x=0.5, font=dict(size=18)
    ),
    template=PLOTLY_TEMPLATE,
    height=500,
    showlegend=False
)
fig.show()

crop_counts = df['label'].value_counts().reset_index()
crop_counts.columns = ['Crop', 'Count']
crop_counts

,Crop,Count
0,rice,100
1,maize,100
2,chickpea,100
3,kidneybeans,100
4,pigeonpeas,100
5,mothbeans,100
6,mungbean,100
7,blackgram,100
8,lentil,100
9,pomegranate,100


### 2.3 Feature Distributions Across Crop Categories

In [32]:
# Box plots of all seven predictors stratified by agronomic category
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=[feature_labels[f] for f in features],
    vertical_spacing=0.18,
    horizontal_spacing=0.06
)

positions = [(r+1, c+1) for r in range(2) for c in range(4)]

for i, feat in enumerate(features):
    r, c = positions[i]
    for cat in cat_names:
        fig.add_trace(
            go.Box(
                y=df[df['category'] == cat][feat],
                name=cat,
                marker=dict(color=CAT_COLORS[cat]),
                line=dict(width=1.2),
                boxpoints=False,
                showlegend=False,
                hoverinfo='skip'
            ),
            row=r, col=c
        )

fig.update_layout(
    title=dict(
        text='<b>Figure 2 — Distribution of Key Features Across Crop Categories</b>',
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    template='simple_white',
    font=dict(family='Arial', size=11, color='black'),
    height=750, width=1100, showlegend=False,
    margin=dict(l=50, r=30, t=80, b=50)
)
fig.update_xaxes(showgrid=False, zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor='lightgrey')
fig.show()

### 2.4 Feature Correlation Matrix

In [33]:
# Pearson correlation heatmap across all predictor variables
corr = df[features].corr().round(3)

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=[feature_labels[f] for f in features],
    y=[feature_labels[f] for f in features],
    colorscale='RdYlGn', zmid=0, zmin=-1, zmax=1,
    text=corr.round(2).values,
    texttemplate='%{text}', textfont=dict(size=10),
    hovertemplate='<b>%{y}</b> vs <b>%{x}</b><br>Pearson r = %{z:.3f}<extra></extra>',
    colorbar=dict(title=dict(text='Pearson r', side='right'), thickness=15)
))

fig.update_layout(
    title=dict(
        text='<b>Figure 3 — Feature Correlation Matrix</b>',
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    template='simple_white',
    font=dict(family='Arial', size=11, color='black'),
    height=550, width=700,
    margin=dict(l=80, r=30, t=80, b=80)
)
fig.update_xaxes(tickangle=-30, showgrid=False, tickfont=dict(size=10))
fig.update_yaxes(showgrid=False, tickfont=dict(size=10))
fig.show()

### 2.5 Principal Component Analysis (PCA) — 2D Feature Space

In [34]:
# PCA projection to assess linear separability across categories
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler_pca = StandardScaler()
X_scaled_pca = scaler_pca.fit_transform(df[features])

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled_pca)

pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0], 'PC2': X_pca[:, 1],
    'Category': df['category'], 'Crop': df['label']
})

pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100
total_var = pc1_var + pc2_var

fig = px.scatter(
    pca_df, x='PC1', y='PC2',
    color='Category', symbol='Category',
    color_discrete_map=CAT_COLORS,
    hover_data={'Crop': True, 'PC1': ':.2f', 'PC2': ':.2f'},
    opacity=0.75,
    title=(
        f"<b>Figure 4 — PCA Projection of Feature Space</b><br>"
        f"<span style='font-size:12px'>"
        f"PC1 ({pc1_var:.1f}% variance), PC2 ({pc2_var:.1f}% variance) — "
        f"Total: {total_var:.1f}%</span>"
    ),
    template='simple_white', height=550, width=800
)

fig.update_traces(marker=dict(size=6, line=dict(width=0.5, color='white')))
fig.update_layout(
    title=dict(x=0.5, xanchor='center', font=dict(size=18, family='Arial')),
    font=dict(family='Arial', size=11, color='black'),
    legend=dict(title='Crop Category', orientation='h', x=0.5, y=-0.15, xanchor='center'),
    margin=dict(l=50, r=30, t=90, b=70)
)
fig.show()

### 2.6 Per-Feature Violin Plots — Top Crops

In [35]:
# Violin plots comparing rainfall distribution across the six most frequent crops
def plot_feature_kde_overlap(df, feature, top_n_crops=5):
    crops = df['label'].value_counts().nlargest(top_n_crops).index
    fig = go.Figure()

    for i, crop in enumerate(crops):
        vals = df[df['label'] == crop][feature].dropna()
        fig.add_trace(go.Violin(
            y=vals, name=crop, box_visible=True, meanline_visible=True,
            opacity=0.75,
            marker=dict(
                color=px.colors.qualitative.Plotly[i % 10],
                line=dict(width=0.5, color='white')
            )
        ))

    fig.update_layout(
        title=dict(
            text=(
                f"<b>Figure 5 — {feature_labels[feature]} Distribution Across Major Crops</b><br>"
                f"<span style='font-size:12px'>Top {top_n_crops} crops by sample frequency</span>"
            ),
            x=0.5, xanchor='center', font=dict(size=18, family='Arial')
        ),
        template='simple_white', height=450, width=800,
        font=dict(family='Arial', size=11),
        xaxis_title='Crop', yaxis_title=feature_labels[feature],
        legend=dict(orientation='h', x=0.5, y=-0.2, xanchor='center'),
        margin=dict(l=60, r=30, t=90, b=80)
    )
    fig.show()

plot_feature_kde_overlap(df, 'rainfall', top_n_crops=6)

### 2.7 Intra-Category Correlation Patterns

In [36]:
# Per-category Pearson correlation heatmaps to reveal crop-specific interactions
def plot_class_correlations(df, features, categories, n_cols=3):
    n_cats = len(categories)
    n_rows = (n_cats + n_cols - 1) // n_cols
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=categories)

    for idx, cat in enumerate(categories):
        subset = df[df['category'] == cat][features]
        corr   = subset.corr().round(2)
        r, c   = divmod(idx, n_cols)

        fig.add_trace(
            go.Heatmap(
                z=corr.values, x=features, y=features,
                colorscale='RdBu_r', zmid=0, showscale=False,
                text=corr.values, texttemplate='%{text}', textfont=dict(size=9)
            ),
            row=r+1, col=c+1
        )

    fig.update_layout(
        title=dict(
            text=(
                "<b>Figure 6 — Feature Correlation Patterns by Crop Category</b><br>"
                "<span style='font-size:12px'>Pearson correlations within each category</span>"
            ),
            x=0.5, xanchor='center', font=dict(size=18, family='Arial')
        ),
        template='simple_white', height=420*n_rows, width=900,
        font=dict(family='Arial', size=10),
        margin=dict(l=50, r=30, t=90, b=50)
    )
    fig.show()

plot_class_correlations(df, features, cat_names)

### 2.8 Soil Fertility Index

In [37]:
# Derived composite nutrient index: SFI = (N + P + K) / 3
df['soil_fertility_index'] = (df['N'] + df['P'] + df['K']) / 3
df_derived = df.copy()

fig = px.box(
    df_derived, x='category', y='soil_fertility_index',
    color='category', color_discrete_map=CAT_COLORS,
    template='simple_white',
    labels={'soil_fertility_index': 'Soil Fertility Index'},
    height=450, width=800
)

fig.update_layout(
    title=dict(
        text=(
            "<b>Figure 7 — Soil Fertility Index Across Crop Categories</b><br>"
            "<span style='font-size:12px'>(N + P + K) / 3 composite nutrient index</span>"
        ),
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    font=dict(family='Arial', size=11),
    legend=dict(orientation='h', x=0.5, y=-0.2, xanchor='center'),
    margin=dict(l=60, r=40, t=90, b=80)
)
fig.show()

### 2.9 Preliminary Permutation Feature Importance

In [38]:
# Quick Random Forest fit on scaled data to compute permutation importance
from sklearn.inspection import permutation_importance

_rf_eda  = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
_le_eda  = LabelEncoder()
_y_eda   = _le_eda.fit_transform(df['label'])
_sc_eda  = StandardScaler()
_Xs_eda  = _sc_eda.fit_transform(df[features])

_rf_eda.fit(_Xs_eda, _y_eda)

perm_eda = permutation_importance(_rf_eda, _Xs_eda, _y_eda,
                                   n_repeats=10, random_state=SEED, n_jobs=1)

imp_df = pd.DataFrame({
    'Feature':    [feature_labels[f] for f in features],
    'Importance': perm_eda.importances_mean,
    'Std':        perm_eda.importances_std
}).sort_values('Importance', ascending=True)

fig = px.bar(
    imp_df, x='Importance', y='Feature', orientation='h', error_x='Std',
    color='Importance', color_continuous_scale='Viridis',
    template='simple_white', height=450, width=850
)

fig.update_layout(
    title=dict(
        text=(
            "<b>Figure 8 — Permutation Feature Importance (Random Forest — EDA)</b><br>"
            "<span style='font-size:12px'>Higher values indicate stronger predictive contribution</span>"
        ),
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    font=dict(family='Arial', size=11),
    coloraxis_showscale=False,
    margin=dict(l=80, r=30, t=90, b=70)
)
fig.update_traces(marker=dict(line=dict(width=0.5, color='black')))
fig.show()

---
<a id="sec3"></a>
## Section 3 — Pre-processing & Feature Engineering

Raw features are encoded, standardised, and partitioned prior to model training.
Two interaction terms are further derived to capture nutrient balance and
combined climatic effects.

### 3.1 Encoding, Scaling, and Train–Test Split

In [39]:
# Encode target labels into integers and apply stratified 70/30 split
label_encoder = LabelEncoder()
df['label_enc'] = label_encoder.fit_transform(df['label'])
class_names = label_encoder.classes_

X = df[features].values
y = df['label_enc'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

# Standardise features to zero mean and unit variance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

y_test_categories = [crop_to_cat[class_names[i]] for i in y_test]

print("=" * 55)
print("DATA PREPROCESSING SUMMARY")
print("=" * 55)
print(f"  Training samples        : {X_train.shape[0]:,}")
print(f"  Testing samples         : {X_test.shape[0]:,}")
print(f"  Number of features      : {X_train.shape[1]}")
print(f"  Number of crop classes  : {len(class_names)}")
print(f"  Number of categories    : {len(cat_names)}")
print(f"  Scaling method          : StandardScaler (μ=0, σ=1)")
print(f"  Train-test split ratio  : 70/30 (stratified)")
print(f"  Random seed             : {SEED}")

DATA PREPROCESSING SUMMARY
  Training samples        : 1,540
  Testing samples         : 660
  Number of features      : 7
  Number of crop classes  : 22
  Number of categories    : 6
  Scaling method          : StandardScaler (μ=0, σ=1)
  Train-test split ratio  : 70/30 (stratified)
  Random seed             : 42


### 3.2 Feature Engineering

In [40]:
# Construct two composite interaction features from domain knowledge
df['NPK_ratio']           = df['N'] / (df['P'] + df['K'] + 1e-6)
df['temp_humidity_index'] = (df['temperature'] * df['humidity']) / 100.0

print("Engineered Features:")
print("  NPK_ratio           — Nitrogen relative to combined P and K nutrient balance")
print("  temp_humidity_index — Composite index of temperature–humidity interaction")
print()
display(df[['NPK_ratio', 'temp_humidity_index']].describe().round(4))

Engineered Features:
  NPK_ratio           — Nitrogen relative to combined P and K nutrient balance
  temp_humidity_index — Composite index of temperature–humidity interaction



,NPK_ratio,temp_humidity_index
count,2200.0000,2200.0000
mean,0.6801,18.5423
std,0.5785,6.9937
min,0.0000,2.4761
25%,0.2179,14.7956
50%,0.4916,19.2788
75%,1.0251,22.5575
max,2.7800,40.7316


---
<a id="sec4"></a>
## Section 4 — Multi-Objective Model Training

Four classifiers are instantiated, each occupying a distinct position on the
**accuracy–interpretability Pareto frontier**:

| Model | Paradigm | XAI Suitability |
|:------|:---------|:----------------|
| **Random Forest** | Ensemble bagging | High — tree-based, SHAP-compatible |
| **XGBoost** | Gradient boosting | High — SHAP TreeExplainer native |
| **Support Vector Machine** | Kernel margin maximisation | Moderate — permutation-based attribution |
| **Artificial Neural Network** | Deep learning (MLP) | Low — black-box, requires proxy methods |

In [41]:
# Instantiate all candidate classifiers with carefully tuned hyperparameters
model_defs = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_features='sqrt',
        class_weight='balanced', random_state=SEED, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss', random_state=SEED, verbosity=0
    ),
    'Support Vector Machine': SVC(
        kernel='rbf', C=10, gamma='scale',
        class_weight='balanced', probability=True, random_state=SEED
    ),
    'Artificial Neural Network': MLPClassifier(
        hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam',
        alpha=1e-4, batch_size=32, learning_rate='adaptive',
        max_iter=500, early_stopping=True, validation_fraction=0.1,
        random_state=SEED
    )
}

trained_models  = {}
training_times  = {}
model_names     = list(model_defs.keys())

print("Model training in progress...\n")

for name, model in model_defs.items():
    start_time = time.time()
    model.fit(X_train_scaled, y_train)
    elapsed = time.time() - start_time
    trained_models[name] = model
    training_times[name] = elapsed
    print(f"  {name:<28} completed in {elapsed:.2f}s")

print("\nAll models trained successfully.")

Model training in progress...

  Random Forest                completed in 0.95s
  XGBoost                      completed in 4.68s
  Support Vector Machine       completed in 0.56s
  Artificial Neural Network    completed in 4.18s

All models trained successfully.


---
<a id="sec5"></a>
## Section 5 — Comparative Performance Evaluation

Each model is assessed on five standard classification metrics computed on the
held-out test set, followed by a 5-fold stratified cross-validation to evaluate
generalisation stability.

### 5.1 Test-Set Metrics (Table 1)

In [42]:
# Compute accuracy, precision, recall, F1, and macro AUC-ROC for all models
results = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)
    y_bin  = label_binarize(y_test, classes=np.arange(len(class_names)))

    results[name] = {
        'Accuracy':          round(accuracy_score(y_test, y_pred), 4),
        'Precision':         round(precision_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'Recall':            round(recall_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'F1-Score':          round(f1_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'AUC-ROC':           round(roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro'), 4),
        'Training Time (s)': round(training_times[name], 2),
        'y_pred':            y_pred,
        'y_prob':            y_prob,
        'y_pred_cats':       [crop_to_cat[class_names[i]] for i in y_pred]
    }

metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'Training Time (s)']
metrics_df  = pd.DataFrame(
    {model: {m: results[model][m] for m in metric_cols} for model in results}
).T

print("=" * 70)
print("TABLE 1 — COMPARATIVE MODEL PERFORMANCE (TEST SET)")
print("=" * 70)

styled_table = (
    metrics_df.style
    .format("{:.4f}")
    .background_gradient(cmap='Greens', subset=['Accuracy','Precision','Recall','F1-Score','AUC-ROC'])
    .background_gradient(cmap='Reds_r',  subset=['Training Time (s)'])
    .highlight_max(subset=['Accuracy','F1-Score','AUC-ROC'], color='#c8e6c9')
    .highlight_min(subset=['Training Time (s)'],             color='#c8e6c9')
    .set_properties(**{'color':'black','font-family':'Arial','font-size':'11pt','text-align':'center'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('font-family','Arial'),('font-size','12pt'),
                  ('text-align','center'),('font-weight','bold'),('color','black')]
    }])
)
display(styled_table)

TABLE 1 — COMPARATIVE MODEL PERFORMANCE (TEST SET)


,Accuracy,Precision,Recall,F1-Score,AUC-ROC,Training Time (s)
Random Forest,0.9939,0.9942,0.9939,0.9939,1.0000,0.9500
XGBoost,0.9909,0.9916,0.9909,0.9909,1.0000,4.6800
Support Vector Machine,0.9924,0.9929,0.9924,0.9924,1.0000,0.5600
Artificial Neural Network,0.9864,0.9870,0.9864,0.9863,0.9999,4.1800


### 5.2 Grouped Bar Chart — Performance Comparison

In [43]:
# Grouped bar chart comparing all five performance metrics across models
fig = go.Figure()
metric_display = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']

for name in model_names:
    color  = MODEL_COLORS.get(name, '#888888')
    values = [results[name][m] for m in metric_display]

    fig.add_trace(go.Bar(
        name=name, x=metric_display, y=values,
        marker=dict(color=color, line=dict(width=0.6, color='black')),
        text=[f'{v:.4f}' for v in values],
        textposition='outside',
        textfont=dict(size=11, color='black'),
        hovertemplate=f'<b>{name}</b><br>%{{x}}: %{{y:.4f}}<extra></extra>'
    ))

fig.update_layout(
    title=dict(
        text=(
            "<b>Figure 9 — Comparative Performance of Machine Learning Models</b><br>"
            "<span style='font-size:12px'>Evaluation across Accuracy, Precision, Recall, F1-Score, and AUC-ROC</span>"
        ),
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    barmode='group', template='simple_white',
    font=dict(family='Arial', size=11, color='black'),
    xaxis=dict(title='Evaluation Metrics', tickfont=dict(size=11), title_font=dict(size=13)),
    yaxis=dict(title='Performance Score', range=[0.88, 1.02],
               tickfont=dict(size=11), title_font=dict(size=13)),
    legend=dict(title='Model', orientation='h', x=0.5, xanchor='center', y=-0.22),
    margin=dict(l=60, r=40, t=95, b=80),
    height=520, width=850
)
fig.show()

### 5.3 Five-Fold Stratified Cross-Validation

In [44]:
# Assess generalisation stability via 5-fold stratified cross-validation
from sklearn.model_selection import cross_val_score, StratifiedKFold

print("Running 5-Fold Stratified Cross-Validation...\n")
cv_results = {}
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, model in trained_models.items():
    print(f"  Evaluating {name}...")
    scores = cross_val_score(model, X_train_scaled, y_train,
                              cv=kf, scoring='accuracy', n_jobs=1)
    cv_results[name] = scores
    print(f"    Mean Accuracy : {scores.mean():.4f}")
    print(f"    Std Dev       : {scores.std():.4f}")
    print("  " + "-" * 38)

Running 5-Fold Stratified Cross-Validation...

  Evaluating Random Forest...
    Mean Accuracy : 0.9948
    Std Dev       : 0.0053
  --------------------------------------
  Evaluating XGBoost...
    Mean Accuracy : 0.9935
    Std Dev       : 0.0046
  --------------------------------------
  Evaluating Support Vector Machine...
    Mean Accuracy : 0.9792
    Std Dev       : 0.0078
  --------------------------------------
  Evaluating Artificial Neural Network...
    Mean Accuracy : 0.9623
    Std Dev       : 0.0180
  --------------------------------------


### 5.4 Cross-Validation Box Plots

In [45]:
# Box plots of fold-level accuracy scores with individual data points overlaid
fig = go.Figure()

for name in model_names:
    scores = cv_results[name]
    color  = MODEL_COLORS.get(name, '#888888')

    fig.add_trace(go.Box(
        y=scores, name=name,
        boxmean=True, boxpoints='all', jitter=0.35, pointpos=0,
        marker=dict(color=color, size=7, line=dict(width=0.5, color='black')),
        line=dict(width=1.2),
        hovertemplate=f'<b>{name}</b><br>Fold Score: %{{y:.4f}}<extra></extra>'
    ))

fig.update_layout(
    title=dict(
        text=(
            "<b>Figure 10 — Cross-Validation Performance Stability</b><br>"
            "<span style='font-size:12px'>5-Fold accuracy distribution across machine learning models</span>"
        ),
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    template='simple_white', font=dict(family='Arial', size=11, color='black'),
    yaxis=dict(title='Cross-Validation Accuracy', tickformat='.3f', range=[0.85, 1.02]),
    xaxis=dict(title='Machine Learning Models'),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.20),
    margin=dict(l=60, r=40, t=95, b=80),
    height=500, width=850
)
fig.show()

### 5.5 Category-Level Confusion Matrices

In [46]:
# Derive category-level prediction labels for all models
y_test_cats = [crop_to_cat[class_names[i]] for i in y_test]

for name in model_names:
    results[name]['y_pred_cats'] = [
        crop_to_cat[class_names[i]] for i in results[name]['y_pred']
    ]

In [47]:
# 2×2 subplot grid of confusion matrices at the agronomic category level
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f"<b>{name}</b><br><span style='font-size:10px'>Accuracy = {results[name]['Accuracy']:.3f}</span>"
        for name in model_names
    ],
    horizontal_spacing=0.20, vertical_spacing=0.22
)

for idx, name in enumerate(model_names):
    r  = idx // 2 + 1
    c  = idx % 2 + 1
    cm = confusion_matrix(y_test_cats, results[name]['y_pred_cats'], labels=cat_names)

    fig.add_trace(
        go.Heatmap(
            z=cm, x=cat_names, y=cat_names, colorscale='Blues',
            text=cm, texttemplate='%{text}', textfont=dict(size=13, color='black'),
            showscale=True if idx == 3 else False,
            colorbar=dict(title='Count', thickness=16, len=0.70, x=1.06),
            hovertemplate=(
                f'<b>{name}</b><br>True: %{{y}}<br>Predicted: %{{x}}<br>'
                'Count: %{z}<extra></extra>'
            )
        ),
        row=r, col=c
    )

fig.update_layout(
    title=dict(
        text=(
            "<b>Figure 11 — Category-Level Confusion Matrices</b><br>"
            "<span style='font-size:12px'>Rows = true categories; Columns = predicted categories</span>"
        ),
        x=0.5, xanchor='center', font=dict(size=18, family='Arial')
    ),
    template='simple_white', font=dict(family='Arial', size=11, color='black'),
    height=820, width=1080, margin=dict(l=80, r=120, t=150, b=80)
)

for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(title_text='Predicted Category', row=i, col=j,
                         tickangle=-25, tickfont=dict(size=11))
        fig.update_yaxes(title_text='True Category', row=i, col=j,
                         tickfont=dict(size=11))
fig.show()

### 5.6 Per-Category F1-Score Heatmap

In [48]:
# Compute per-category weighted F1 scores for all models
per_category_f1 = {}

for model_name in model_names:
    report = classification_report(
        y_test_cats, results[model_name]['y_pred_cats'],
        labels=cat_names, output_dict=True, zero_division=0
    )
    per_category_f1[model_name] = {
        cat: report[cat]['f1-score'] for cat in cat_names
    }

f1_matrix = pd.DataFrame(per_category_f1).T

fig = go.Figure(go.Heatmap(
    z=f1_matrix.values, x=cat_names, y=model_names,
    text=f1_matrix.round(4).values,
    texttemplate='<b>%{text}</b>', textfont=dict(size=14),
    colorscale='RdYlGn', zmin=0.95, zmax=1.00,
    colorbar=dict(title='F1-Score', thickness=16, len=0.85),
    hovertemplate='<b>Model:</b> %{y}<br><b>Category:</b> %{x}<br><b>F1:</b> %{z:.4f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text=(
            "<b>Figure 12 — Per-Category F1-Score Heatmap</b><br>"
            "<sup>Comparative classification performance across agronomic crop categories</sup>"
        ),
        x=0.5, font=dict(size=18)
    ),
    template=PLOTLY_TEMPLATE, height=360,
    xaxis=dict(title='Crop Category', tickfont=dict(size=11)),
    yaxis=dict(title='Machine Learning Model', tickfont=dict(size=11)),
    margin=dict(l=80, r=60, t=90, b=60)
)
fig.show()

---
<a id="sec6"></a>
## Section 6 — Global Explainability Analysis (SHAP)

SHAP (SHapley Additive exPlanations) grounds feature attribution in cooperative
game theory, offering both globally consistent and locally accurate explanations.
TreeExplainer is applied to tree-based models; permutation importance is used
as a model-agnostic surrogate for SVM and ANN.